# Model Explanation Deep Dive
This notebook continues the root cause analysis by focusing on the Random Forest model's explanations, including feature importance and partial dependence plots.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.inspection import PartialDependenceDisplay

# Load dataset
df = pd.read_csv('Egencia Analytics Case Study.csv', parse_dates=['BOOKING_DATE', 'TRAVEL_START'], dayfirst=True)
df.head()

## Preprocessing & Feature Engineering
- Filter only 'Reserve' bookings
- Compute lead time
- Encode categorical features

In [ ]:
# Filter and drop missing
df = df[df['BOOKING_TYPE_NAME'] == 'Reserve'].dropna(subset=['PRODUCT_NAME','PLATFORM','BOOKING_DATE','TRAVEL_START','TRANSACTION_COUNT','BOOKING_AMOUNT'])
# Lead time
df['LEAD_TIME_DAYS'] = (df['TRAVEL_START'] - df['BOOKING_DATE']).dt.days
# Encode categoricals
le_prod = LabelEncoder(); le_plat = LabelEncoder()
df['P_ENC'] = le_prod.fit_transform(df['PRODUCT_NAME'])
df['PLT_ENC'] = le_plat.fit_transform(df['PLATFORM'])

## Model Training
Train a Random Forest to predict booking amount.

In [ ]:
# Features and target
X = df[['P_ENC','PLT_ENC','LEAD_TIME_DAYS','TRANSACTION_COUNT']]
y = df['BOOKING_AMOUNT']

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Train model
rf = RandomForestRegressor(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)

## Feature Importance
Plot feature importances from the Random Forest model.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

importances = rf.feature_importances_
feat_names = X_train.columns
importance_df = pd.DataFrame({'Feature': feat_names, 'Importance': importances}).sort_values(by='Importance', ascending=False)

# Plot
plt.figure(figsize=(8,4))
sns.barplot(data=importance_df, x='Importance', y='Feature')
plt.title('Feature Importance')
plt.tight_layout()
plt.show()

## Partial Dependence Plots
Visualize the effect of Transaction Count and Lead Time on predicted booking amount.

In [ ]:
from sklearn.inspection import PartialDependenceDisplay
fig, ax = plt.subplots(1, 2, figsize=(12,5))
PartialDependenceDisplay.from_estimator(rf, X_train, ['TRANSACTION_COUNT'], ax=ax[0])
ax[0].set_title('Partial Dependence: Transaction Count')
PartialDependenceDisplay.from_estimator(rf, X_train, ['LEAD_TIME_DAYS'], ax=ax[1])
ax[1].set_title('Partial Dependence: Lead Time (days)')
plt.tight_layout()
plt.show()

In [ ]:
nbf.write(nb, 'booking_model_explanation.ipynb')